# 1장 · 트랜스포머 큰 그림 — 다음 단어 맞히기 🎯

웹 챕터 **"LLM은 거대한 다음 단어 맞히기 게임"** 과 짝을 이루는 실습입니다.
웹의 게임에서 손으로 단어를 이어붙였다면, 여기서는 **자기회귀 생성 루프**를 직접 코드로 돌립니다.

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요
from tutor import chat

chat('ch1')

## 1. 아주 작은 목업 "언어 모델"

진짜 LLM은 수만 개 단어에 점수를 매기지만, 원리를 보려고 **손바닥만 한 모델**을 씁니다.
`MODEL[마지막 단어] = [(다음 후보 단어, 점수), ...]` 형태의 표예요.
표에 없는 단어(문장을 끝맺는 말)는 마침표 `.` 만 후보로 내놓습니다. (웹 게임과 동일)

In [1]:
import numpy as np

SEED = '나는'
END = '.'

MODEL = {
    '나는':   [('밥을', 3.0), ('커피를', 2.0), ('학교에', 1.5), ('영화를', 0.5)],
    '밥을':   [('먹었다', 3.5), ('남겼다', 0.5)],
    '커피를': [('마셨다', 3.5), ('쏟았다', 0.3)],
    '학교에': [('갔다', 3.5), ('늦었다', 1.0)],
    '영화를': [('봤다', 3.5), ('예매했다', 1.0)],
}

def softmax(z):
    z = np.asarray(z, dtype=float); z = z - z.max()
    e = np.exp(z); return e / e.sum()

def candidates(last):
    # 표에 없으면 마침표만 후보 → 문장을 끝맺는다
    return MODEL.get(last, [(END, 5.0)])

## 2. 다음 단어 하나 예측해 보기

마지막 단어가 `'나는'` 일 때, 후보 단어들의 **확률**을 봅시다. (웹 게임의 % 막대와 같은 값)

In [2]:
last = '나는'
cands = candidates(last)
words = [w for w, _ in cands]
logits = [l for _, l in cands]
probs = softmax(logits)

for w, p in zip(words, probs):
    print(f'{w:5s} : {p*100:5.1f}%')
print('가장 확률 높은 단어 →', words[int(np.argmax(probs))])

밥을    :  59.8%
커피를   :  22.0%
학교에   :  13.3%
영화를   :   4.9%
가장 확률 높은 단어 → 밥을


## 3. 자기회귀 생성 루프 (핵심!)

**다음 단어 고르기 → 문장에 붙이기 → 다시 예측**. 마침표가 나올 때까지 반복합니다.
여기선 항상 **가장 확률 높은 단어**를 고르는 방식(그리디, greedy)으로 돌립니다.

In [3]:
def generate(seed=SEED, verbose=True):
    sentence = [seed]
    while sentence[-1] != END:
        cands = candidates(sentence[-1])
        words = [w for w, _ in cands]
        probs = softmax([l for _, l in cands])
        nxt = words[int(np.argmax(probs))]   # 그리디: 최댓값 선택
        if verbose:
            shown = ' '.join(w for w in sentence if w != END)
            print(f"[{shown}] 다음 → '{nxt}' ({probs.max()*100:.0f}%)")
        sentence.append(nxt)
    text = ' '.join(w for w in sentence if w != END) + '.'
    return text

print('완성:', generate())

[나는] 다음 → '밥을' (60%)
[나는 밥을] 다음 → '먹었다' (95%)
[나는 밥을 먹었다] 다음 → '.' (100%)
완성: 나는 밥을 먹었다.


## 4. 확률적으로 골라 보기 (매번 달라지는 답)

항상 1등만 고르면 답이 늘 똑같습니다. **확률에 따라 살짝 무작위로** 고르면
같은 시작에서도 매번 다른 문장이 나와요. LLM이 매번 조금씩 다르게 답하는 원리입니다.
(고르는 방법은 12장에서 자세히.)

In [4]:
rng = np.random.default_rng(0)   # 재현 위해 시드 고정

def generate_sampling(seed=SEED):
    sentence = [seed]
    while sentence[-1] != END:
        cands = candidates(sentence[-1])
        words = [w for w, _ in cands]
        probs = softmax([l for _, l in cands])
        nxt = rng.choice(words, p=probs)     # 확률에 따라 무작위 선택
        sentence.append(nxt)
    return ' '.join(w for w in sentence if w != END) + '.'

for i in range(5):
    print(f'{i+1}번째:', generate_sampling())

1번째: 나는 커피를 마셨다.
2번째: 나는 밥을 먹었다.
3번째: 나는 커피를 마셨다.
4번째: 나는 학교에 갔다.
5번째: 나는 학교에 갔다.


## 5. 직접 해보기 ✏️

`MODEL` 에 새 갈래를 추가해서 문장이 더 길어지게 만들어 보세요.
예: `'늦었다'` 다음에 `('지만', ...)` 같은 걸 잇는 식. (정답은 없어요. 자유롭게!)

아래는 최소 확인: 그리디 생성이 항상 마침표로 끝나는지 검사합니다.

In [5]:
out = generate(verbose=False)
print('생성 결과:', out)
assert out.endswith('.'), '문장은 마침표로 끝나야 합니다!'
assert '나는' in out, '시작 단어가 들어 있어야 합니다!'
print('✅ 자기회귀 생성이 마침표로 잘 끝났습니다.')

생성 결과: 나는 밥을 먹었다.
✅ 자기회귀 생성이 마침표로 잘 끝났습니다.


---
이번엔 "다음 단어를 확률로 골라 이어붙인다"는 **큰 그림**을 코드로 확인했습니다.
다음 챕터(ch2)부터는 이 그림 속 각 칸(**토큰화·임베딩·어텐션…**)을 하나씩 확대합니다. 🔍